In [ ]:
from urllib3.http2.probe import acquire_and_get
from xgboost.sklearn import XGBRegressor
from bayes_opt import BayesianOptimization
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
import xgboost as xgb
import warnings
import seaborn as sns

warnings.filterwarnings('ignore')

random_state = 1

def CV_group(data):
    if data.isnull().values.any():
        print("警告：输入数据包含NaN值，将用中位数填充")
        print("列NaN统计：")
        print(data.isnull().sum()[data.isnull().sum() > 0])
        
        numeric_cols = data.select_dtypes(include=[np.number]).columns
        for col in numeric_cols:
            if data[col].isnull().any():
                median_val = data[col].median()
                data[col].fillna(median_val, inplace=True)
                print(f"列 '{col}'：用中位数 {median_val:.4f} 填充了 {data[col].isnull().sum()} 个NaN值")
    
    data['ref_FO'] = data['ID'].map(str)
    ref_JQ = data['ID']
    tt_ion = pd.DataFrame(ref_JQ.value_counts(ascending=False))
    tt_ion_index = tt_ion.index

    one = pd.DataFrame()
    two = pd.DataFrame()
    three = pd.DataFrame()
    four = pd.DataFrame()
    five = pd.DataFrame()
    for i in range(len(tt_ion_index)):
        a = [len(one), len(two), len(three), len(four), len(five)]
        if a.index(min(a)) == 0:
            one = pd.concat([one, pd.DataFrame(data[data['ID'] == tt_ion_index[i]])])
        elif a.index(min(a)) == 1:
            two = pd.concat([two, pd.DataFrame(data[data['ID'] == tt_ion_index[i]])])
        elif a.index(min(a)) == 2:
            three = pd.concat([three, pd.DataFrame(data[data['ID'] == tt_ion_index[i]])])
        elif a.index(min(a)) == 3:
            four = pd.concat([four, pd.DataFrame(data[data['ID'] == tt_ion_index[i]])])
        elif a.index(min(a)) == 4:
            five = pd.concat([five, pd.DataFrame(data[data['ID'] == tt_ion_index[i]])])

    print('one len: ', len(one))
    print('two len: ', len(two))
    print('three len: ', len(three))
    print('four len: ', len(four))
    print('five len: ', len(five))

    return one, two, three, four, five


class ML_model():
    def __init__(self, train, test, one, two, three, four, five, features):
        self._check_and_fill_data(train, "训练集")
        self._check_and_fill_data(test, "测试集")
        self._check_and_fill_data(one, "分组1")
        self._check_and_fill_data(two, "分组2")
        self._check_and_fill_data(three, "分组3")
        self._check_and_fill_data(four, "分组4")
        self._check_and_fill_data(five, "分组5")
        
        self.X_train = pd.DataFrame(train)
        self.X_test = pd.DataFrame(test)
        self.y_train = train['Js']
        self.y_test = test['Js']
        self.X_train_1 = pd.concat([two, three, four, five], join='outer')
        self.X_test_1 = one
        self.X_train_2 = pd.concat([one, three, four, five], join='outer')
        self.X_test_2 = two
        self.X_train_3 = pd.concat([one, two, four, five], join='outer')
        self.X_test_3 = three
        self.X_train_4 = pd.concat([one, two, three, five], join='outer')
        self.X_test_4 = four
        self.X_train_5 = pd.concat([one, two, three, four], join='outer')
        self.X_test_5 = five
        self.features = features
        
        self._fill_cv_datasets()
        
        self.y_train_1 = self._safe_get_target(self.X_train_1, 'Js', '分组1训练集')
        self.y_train_2 = self._safe_get_target(self.X_train_2, 'Js', '分组2训练集')
        self.y_train_3 = self._safe_get_target(self.X_train_3, 'Js', '分组3训练集')
        self.y_train_4 = self._safe_get_target(self.X_train_4, 'Js', '分组4训练集')
        self.y_train_5 = self._safe_get_target(self.X_train_5, 'Js', '分组5训练集')
        self.y_test_1 = self._safe_get_target(self.X_test_1, 'Js', '分组1测试集')
        self.y_test_2 = self._safe_get_target(self.X_test_2, 'Js', '分组2测试集')
        self.y_test_3 = self._safe_get_target(self.X_test_3, 'Js', '分组3测试集')
        self.y_test_4 = self._safe_get_target(self.X_test_4, 'Js', '分组4测试集')
        self.y_test_5 = self._safe_get_target(self.X_test_5, 'Js', '分组5测试集')
        
        self.X_train_1 = self._safe_select_features(self.X_train_1, "分组1训练集")
        self.X_train_2 = self._safe_select_features(self.X_train_2, "分组2训练集")
        self.X_train_3 = self._safe_select_features(self.X_train_3, "分组3训练集")
        self.X_train_4 = self._safe_select_features(self.X_train_4, "分组4训练集")
        self.X_train_5 = self._safe_select_features(self.X_train_5, "分组5训练集")
        self.X_test_1 = self._safe_select_features(self.X_test_1, "分组1测试集")
        self.X_test_2 = self._safe_select_features(self.X_test_2, "分组2测试集")
        self.X_test_3 = self._safe_select_features(self.X_test_3, "分组3测试集")
        self.X_test_4 = self._safe_select_features(self.X_test_4, "分组4测试集")
        self.X_test_5 = self._safe_select_features(self.X_test_5, "分组5测试集")
    
    def _check_and_fill_data(self, data, name):
        """检查数据中的NaN值，用中位数填充而不是删除"""
        if data is None or len(data) == 0:
            return
            
        nan_count = data.isnull().sum().sum()
        if nan_count > 0:
            print(f"\n{name} 包含 {nan_count} 个NaN值")
            print(f"{name} 列NaN统计：")
            nan_cols = data.isnull().sum()[data.isnull().sum() > 0]
            print(nan_cols)
            
            numeric_cols = data.select_dtypes(include=[np.number]).columns
            for col in numeric_cols:
                if data[col].isnull().any():
                    median_val = data[col].median()
                    filled_count = data[col].isnull().sum()
                    data[col].fillna(median_val, inplace=True)
                    print(f"{name} - 列 '{col}'：用中位数 {median_val:.4f} 填充了 {filled_count} 个NaN值")
    
    def _fill_cv_datasets(self):
        """填充交叉验证数据集中的NaN值"""
        datasets = [
            (self.X_train_1, "训练集1"), (self.X_test_1, "测试集1"),
            (self.X_train_2, "训练集2"), (self.X_test_2, "测试集2"),
            (self.X_train_3, "训练集3"), (self.X_test_3, "测试集3"),
            (self.X_train_4, "训练集4"), (self.X_test_4, "测试集4"),
            (self.X_train_5, "训练集5"), (self.X_test_5, "测试集5")
        ]
        
        for dataset, name in datasets:
            if dataset is None or len(dataset) == 0:
                continue
                
            nan_count = dataset.isnull().sum().sum()
            if nan_count > 0:
                print(f"\n{name} 包含 {nan_count} 个NaN值")
                nan_cols = dataset.isnull().sum()[dataset.isnull().sum() > 0]
                print(f"{name} NaN列：")
                print(nan_cols)
                
                numeric_cols = dataset.select_dtypes(include=[np.number]).columns
                for col in numeric_cols:
                    if dataset[col].isnull().any():
                        median_val = dataset[col].median()
                        filled_count = dataset[col].isnull().sum()
                        dataset[col].fillna(median_val, inplace=True)
                        print(f"{name} - 列 '{col}'：用中位数 {median_val:.4f} 填充了 {filled_count} 个NaN值")
    
    def _safe_get_target(self, data, target_col, name):
        """安全获取目标变量，用中位数填充NaN"""
        if target_col not in data.columns:
            raise ValueError(f"{name} 缺少目标列 '{target_col}'")
        
        y = data[target_col].copy()
        
        if y.isnull().any():
            print(f"\n{name} 的目标变量包含NaN值")
            nan_count = y.isnull().sum()
            median_val = y.median()
            y.fillna(median_val, inplace=True)
            print(f"目标变量中 {nan_count} 个NaN值，将用中位数 {median_val:.4f} 填充")
        
        return y
    
    def _safe_select_features(self, data, name):
        """安全选择特征，用中位数填充NaN"""
        missing_features = [f for f in self.features if f not in data.columns]
        if missing_features:
            raise ValueError(f"{name} 缺少特征列: {missing_features}")
        
        X = data[self.features].copy()
        
        if X.isnull().values.any():
            print(f"\n{name} 的特征数据包含NaN值")
            nan_cols = X.isnull().sum()[X.isnull().sum() > 0]
            print(f"{name} 特征NaN统计：")
            print(nan_cols)
            
            for col in X.columns:
                if X[col].isnull().any():
                    median_val = X[col].median()
                    filled_count = X[col].isnull().sum()
                    X[col].fillna(median_val, inplace=True)
                    print(f"{name} - 特征 '{col}'：用中位数 {median_val:.4f} 填充了 {filled_count} 个NaN值")
        
        if np.isinf(X.values).any():
            print(f"\n{name} 的特征数据包含无穷大值，将替换为中位数")
            for col in X.columns:
                if np.isinf(X[col]).any():
                    median_val = X[col][np.isfinite(X[col])].median()
                    X[col].replace([np.inf, -np.inf], median_val, inplace=True)
                    print(f"{name} - 特征 '{col}'：无穷大值替换为中位数 {median_val:.4f}")
        
        return X

    def rem_name(self, org_data):
        if 'ref_ID' not in org_data.columns:
            print("警告：数据缺少'ref_ID'列，将创建默认标识")
            return pd.DataFrame(index=org_data.index)
        
        data_name = org_data.loc[:, ['ref_ID']]
        return data_name

    def CV_train(self, regressor, X_train, y_train, X_test, y_test):
        if X_train.isnull().values.any() or y_train.isnull().values.any() or X_test.isnull().values.any() or y_test.isnull().values.any():
            print("警告：训练/测试数据包含NaN值，将用中位数填充")
            
            for col in X_train.columns:
                if X_train[col].isnull().any():
                    X_train[col].fillna(X_train[col].median(), inplace=True)
            
            if y_train.isnull().any():
                y_train.fillna(y_train.median(), inplace=True)
            
            for col in X_test.columns:
                if X_test[col].isnull().any():
                    X_test[col].fillna(X_train[col].median(), inplace=True)
            
            if y_test.isnull().any():
                y_test.fillna(y_test.median(), inplace=True)
        
        cv_result = regressor.fit(X_train, y_train)
        cv_y_pred = cv_result.predict(X_test)
        return np.sqrt(mean_squared_error(y_test, cv_y_pred))

    def data_prep_for_model(self, regressor):
        rmse_1 = self.CV_train(regressor, self.X_train_1, self.y_train_1, self.X_test_1, self.y_test_1)
        rmse_2 = self.CV_train(regressor, self.X_train_2, self.y_train_2, self.X_test_2, self.y_test_2)
        rmse_3 = self.CV_train(regressor, self.X_train_3, self.y_train_3, self.X_test_3, self.y_test_3)
        rmse_4 = self.CV_train(regressor, self.X_train_4, self.y_train_4, self.X_test_4, self.y_test_4)
        rmse_5 = self.CV_train(regressor, self.X_train_5, self.y_train_5, self.X_test_5, self.y_test_5)
        rmse = [rmse_1, rmse_2, rmse_3, rmse_4, rmse_5]
        rmse_mean = np.mean(rmse)
        return -1 * rmse_mean

    def xgb_evaluate(self, colsample_bytree, gamma, learning_rate, max_depth, min_child_weight,
                     reg_alpha, reg_lambda, subsample, n_estimators):
        regressor = XGBRegressor(objective='reg:squarederror',
                                 booster='gbtree',
                                 eval_metric='rmse',
                                 colsample_bytree=colsample_bytree,
                                 gamma=gamma,
                                 learning_rate=learning_rate,
                                 max_depth=int(max_depth),
                                 min_child_weight=int(min_child_weight),
                                 reg_alpha=reg_alpha,
                                 reg_lambda=reg_lambda,
                                 subsample=subsample,
                                 n_estimators=int(n_estimators))

        return self.data_prep_for_model(regressor)

    def param(self):
        pbounds = {'colsample_bytree': (0.5, 0.9),
                   'gamma': (0, 1),
                   'learning_rate': (0.0001, 0.1),
                   'max_depth': (3, 10),
                   'min_child_weight': (0.0001, 5),
                   'reg_alpha': (0, 1),
                   'reg_lambda': (0, 1),
                   'subsample': (0.5, 0.9),
                   'n_estimators': (1, 1000)}
        bo = BayesianOptimization(self.xgb_evaluate, pbounds=pbounds, verbose=2, random_state=random_state)

        print('=' * 20)
        print()
        bo.maximize(init_points=30, n_iter=200)
        F_params = pd.DataFrame(bo.max['params'], index=range(1))
        return bo.max['params']


    def final_model(self, X_train, y_train):
        params = self.param()
        
        X_train_clean = X_train.copy()
        y_train_clean = y_train.copy()
        
        if X_train_clean.isnull().values.any():
            print("\n最终模型训练：训练特征包含NaN值，将用中位数填充")
            for col in X_train_clean.columns:
                if X_train_clean[col].isnull().any():
                    median_val = X_train_clean[col].median()
                    X_train_clean[col].fillna(median_val, inplace=True)
                    print(f"特征 '{col}'：用中位数 {median_val:.4f} 填充")
        
        if y_train_clean.isnull().any():
            print("\n最终模型训练：目标变量包含NaN值，将用中位数填充")
            median_val = y_train_clean.median()
            y_train_clean.fillna(median_val, inplace=True)
            print(f"目标变量：用中位数 {median_val:.4f} 填充")
        
        final_model = xgb.XGBRegressor(colsample_bytree=params['colsample_bytree'], gamma=params['gamma'],
                                       learning_rate=params['learning_rate'], max_depth=int(params['max_depth']),
                                       min_child_weight=params['min_child_weight'],
                                       n_estimators=round(params['n_estimators']),
                                       early_stop_rounds=10, reg_alpha=params['reg_alpha'],
                                       reg_lambda=params['reg_lambda'], subsample=params['subsample'],
                                       silent=1,
                                       random_state=random_state,
                                       nthread=-1)
        final_model.fit(X_train_clean, y_train_clean)
        return final_model

    def result(self, model):
        X_train_name = self.rem_name(self.X_train)
        X_test_name = self.rem_name(self.X_test)
        y_train = self.X_train['Js']
        y_test = self.X_test['Js']
        
        X_test_clean = self.X_test[self.features].copy()
        if X_test_clean.isnull().values.any():
            print("\n评估：测试集特征包含NaN值，将用中位数填充")
            for col in X_test_clean.columns:
                if X_test_clean[col].isnull().any():
                    train_median = self.X_train[self.features][col].median()
                    X_test_clean[col].fillna(train_median, inplace=True)
                    print(f"测试集特征 '{col}'：用训练集中位数 {train_median:.4f} 填充")
        
        y_pred = model.predict(X_test_clean)
        y_train_pred = model.predict(self.X_train[self.features])
        train_err = np.sqrt(mean_squared_error(y_train, y_train_pred))

        val_err = np.sqrt(mean_squared_error(y_test, y_pred))
        mae = mean_absolute_error(y_test, y_pred)

        MAE = pd.DataFrame({"Train_RMSE": str(train_err), "Test RMSE": str(val_err), "MAE": str(mae)}, index=range(1))

        print("\tTraining RMSE\t\t" + str(train_err))
        print("\tTest RMSE\t\t" + str(val_err))
        print("\tMean absolute error\t" + str(mae))
        y_pred = pd.DataFrame(y_pred)
        y_pred.columns = ['pred']
        y_test = pd.DataFrame(y_test)
        y_test.columns = ['y_test']
        y_train_pred = pd.DataFrame(y_train_pred)
        y_train_pred.columns = ['pred']
        total = pd.concat([X_train_name, self.X_train[self.features], y_train], axis=1)

        y_train_pred.index = total.index

        total_train = pd.concat([total, y_train_pred], axis=1)

        df_1_test = pd.concat([self.X_test[self.features], y_test], axis=1)
        y_test.index = df_1_test.index

        y_pred.index = df_1_test.index
        y_pred

        total_test = pd.concat([X_test_name, df_1_test, y_pred], axis=1)
        total_train['error'] = total_train['Js'] - total_train['pred']
        total_test['error'] = total_test['y_test'] - total_test['pred']
        train_error = total_train.loc[:, ['Js', 'pred', 'error']]
        test_error = total_test.loc[:, ['y_test', 'pred', 'error']]
        f1 = plt.figure()
        ax = sns.scatterplot(x="y_test", y="pred", data=test_error)
        ax.set(xlim=(0, 110), ylim=(0, 105), title='Test pred-real Js')
        return self

In [ ]:

df = pd.read_excel('FO xgboost.xlsx', sheet_name='data')
df.columns = ['ID', 'MPDC', 'TMCC', 'MPDIPT',
                  'TMCIPT', 'MT', 'PTE', 'PTI', 'OPS',
                  'CT', 'WCA', 'WP', 'MO',
                  'DST', 'DSC', 'ST', 'CV', 'Jw', 'Js', 'R', 'ref_ID']
df.index = range(len(df))
features = ['MPDC', 'TMCC', 'MPDIPT',
                'TMCIPT', 'MT', 'PTE', 'PTI','OPS',
                'CT', 'WCA', 'WP', 'MO',
                'DST', 'DSC', 'ST', 'CV',]
ref_id=df['ref_ID']
train = df[ pd.to_numeric(ref_id) != 1]
test = df[pd.to_numeric(ref_id) == 1]

df['ref_FO'] = df['ref_ID'].map(str)
ref_FO = df['ref_FO']
b = pd.DataFrame(ref_FO.value_counts().sort_values(ascending=False))
ref_FO = b.index
df.dropna(axis=0, inplace=True)
one, two, three, four, five = CV_group(train)
ml_model = ML_model(train, test, one, two, three, four, five, features)
final_model = ml_model.final_model(train[features], train['Js'])
result = ml_model.result(final_model)

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression
from scipy.stats import linregress
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np
import pandas as pd
import plotly.io as pio
import os

train['pred'] = final_model.predict(train[features])
test['pred']  = final_model.predict(test[features])

train_r2   = r2_score(train['Js'], train['pred'])
train_mae  = mean_absolute_error(train['Js'], train['pred'])
train_rmse = np.sqrt(mean_squared_error(train['Js'], train['pred']))

test_r2   = r2_score(test['Js'], test['pred'])
test_mae  = mean_absolute_error(test['Js'], test['pred'])
test_rmse = np.sqrt(mean_squared_error(test['Js'], test['pred']))

train['mark'] = 'Training set'   
test['mark']  = 'Test set'
total_data = pd.concat([train, test])

x_data = test['Js'].values.reshape(-1, 1)
y_data = test['pred'].values.reshape(-1, 1)
model  = LinearRegression(fit_intercept=False)
model.fit(x_data, y_data)
pred_line = model.predict(x_data)

print("\n" + "=" * 50)
print("模型性能评估指标")
print("=" * 50)
print(f"训练集 (Training Set):")
print(f"  R² (决定系数):   {train_r2:.4f}")
print(f"  RMSE (均方根误差): {train_rmse:.4f}")
print(f"  MAE (平均绝对误差): {train_mae:.4f}")
print(f"\n测试集 (Test Set/验证集):")
print(f"  R² (决定系数):   {test_r2:.4f}")
print(f"  RMSE (均方根误差): {test_rmse:.4f}")
print(f"  MAE (平均绝对误差): {test_mae:.4f}")
print("=" * 50 + "\n")


COLOR_TRAIN   = '3C5488'   
COLOR_TEST    = 'E64B35'   
COLOR_REF     = '8C8C8C'   
COLOR_FIT     = 'E64B35'   
COLOR_BG      = 'FFFFFF'   
COLOR_AXIS    = '2D2D2D'   
COLOR_GRID    = 'E8E8E8'   

FONT_FAMILY   = 'Arial'
TITLE_SIZE    = 28           
TICK_SIZE     = 20           
LEGEND_SIZE   = 18           
PLOT_W, PLOT_H = 680, 620
AXIS_RANGE    = [-10, 120]
DTICK         = 25

fig = px.scatter(
    total_data, x='Js', y='pred', color='mark',
    width=PLOT_W, height=PLOT_H,
    range_x=AXIS_RANGE, range_y=AXIS_RANGE,
    color_discrete_map={                       
        'Training set': COLOR_TRAIN,
        'Test set':     COLOR_TEST,
    },
    category_orders={'mark': ['Training set', 'Test set']},  
)

fig.update_layout(
    plot_bgcolor=COLOR_BG,
    paper_bgcolor=COLOR_BG,
    margin=dict(l=80, r=30, t=30, b=80),      
    showlegend=True,
    legend=dict(
        x=0.02, y=0.98,                        
        xanchor='left', yanchor='top',
        bgcolor='rgba(255,255,255,0.85)',       
        bordercolor='CCCCCC',
        borderwidth=1,
        font=dict(family=FONT_FAMILY, size=LEGEND_SIZE, color=COLOR_AXIS),
        itemsizing='constant',
    ),
    font=dict(family=FONT_FAMILY, color=COLOR_AXIS),
)

fig.update_xaxes(
    title_text='Real Js (%)',
    title_font=dict(family=FONT_FAMILY, size=TITLE_SIZE, color=COLOR_AXIS),
    title_standoff=15,
    tickfont=dict(family=FONT_FAMILY, size=TICK_SIZE, color=COLOR_AXIS),
    dtick=DTICK,
    range=AXIS_RANGE,
    showline=True, linewidth=2, linecolor=COLOR_AXIS, mirror=True,
    showgrid=True, gridwidth=1, gridcolor=COLOR_GRID,
    ticks='inside', ticklen=6, tickwidth=1.5, tickcolor=COLOR_AXIS,
    zeroline=False,
)

fig.update_yaxes(
    title_text='Predicted Js (%)',
    title_font=dict(family=FONT_FAMILY, size=TITLE_SIZE, color=COLOR_AXIS),
    title_standoff=12,
    tickfont=dict(family=FONT_FAMILY, size=TICK_SIZE, color=COLOR_AXIS),
    dtick=DTICK,
    range=AXIS_RANGE,
    showline=True, linewidth=2, linecolor=COLOR_AXIS, mirror=True,
    showgrid=True, gridwidth=1, gridcolor=COLOR_GRID,
    ticks='inside', ticklen=6, tickwidth=1.5, tickcolor=COLOR_AXIS,
    zeroline=False,
    scaleanchor='x', scaleratio=1,             
)

fig.update_traces(
    marker=dict(
        size=9,
        opacity=0.78,
        line=dict(width=0.8, color='rgba(255,255,255,0.9)'),  
    ),
    selector=dict(mode='markers'),
)

fig.add_trace(go.Scatter(
    x=[0, 100], y=[0, 100],
    mode='lines',
    name='y = x',
    line=dict(color=COLOR_REF, width=1.8, dash='dash'),
    hoverinfo='skip',
))



fig.show()

In [ ]:
import shap
%matplotlib inline
X_train = train.copy()
X_train_name = X_train.loc[:,['ID']]

X_train = X_train[features]

f1 = plt.figure()
explainer = shap.TreeExplainer(final_model)
shap_values = explainer.shap_values(X_train)
shap.summary_plot(shap_values, X_train, feature_names=X_train.columns, show=False)
shap_values = pd.DataFrame(shap_values)
shap_features = [feature+'-SHAP' for feature in features]
shap_values.columns = shap_features.copy()

X_train_name.index = range(len(X_train_name))
X_train.index = range(len(X_train))
shap_values.index = range(len(shap_values))
total = pd.concat([X_train_name, X_train, shap_values], axis=1)
shap_values.index = X_train.index

import shap
import matplotlib.pyplot as plt
import pandas as pd

%matplotlib inline

X_train = train.copy()
X_train_name = X_train.loc[:,['ID']]
X_train = X_train[features]

f1 = plt.figure(figsize=(12, 8))

explainer = shap.TreeExplainer(final_model)
shap_values = explainer.shap_values(X_train)

shap.summary_plot(shap_values, X_train, feature_names=X_train.columns, show=False)

plt.tight_layout()

output_filename = f"SHAP_summary_plot_{pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')}.png"
plt.savefig(output_filename, dpi=300, bbox_inches='tight')

pdf_filename = output_filename.replace('.png', '.pdf')
plt.savefig(pdf_filename, bbox_inches='tight')

print(f"SHAP摘要图已保存为: {output_filename}")
print(f"PDF版本已保存为: {pdf_filename}")

shap_values_df = pd.DataFrame(shap_values)
shap_features = [feature+'-SHAP' for feature in features]
shap_values_df.columns = shap_features.copy()

X_train_name.index = range(len(X_train_name))
X_train.index = range(len(X_train))
shap_values_df.index = range(len(shap_values_df))
total = pd.concat([X_train_name, X_train, shap_values_df], axis=1)

plt.close(f1)

In [ ]:
shap_values = total.copy()
shap_sum = np.abs(shap_values[shap_features]).mean(axis=0)
importance_df = pd.DataFrame([shap_values[features].columns.tolist(), shap_sum.tolist()]).T
importance_df.columns = ['column_name', 'shap_importance']
importance_df = importance_df.sort_values('shap_importance', ascending=False)
importance_df.index = range(len(shap_sum))
importance_df.sort_values(by='shap_importance',ascending=True, inplace=True)
importance_df.shap_importance = importance_df.shap_importance/importance_df['shap_importance'][0]*100

import plotly.express as px

X_title = 'SHAP importance'
Y_title = ''

title_font_size = 43
tickfont_size = 35
ticklen = 5
tickwidth = 4
title_standoff = 20


plot_width = 700
plot_height = 500

fig = px.bar(importance_df, x="shap_importance", y="column_name", orientation='h',
             width=plot_width, height=plot_height)

for data in fig.data:
    data["width"] = 0.7


fig.update_layout(
    xaxis = dict(
        title_text = X_title,
        title_font = {"size": title_font_size},
        tickfont = dict(family='Arial', color='black', size=tickfont_size),
        title_standoff = title_standoff,
        color='black',
        ticks="inside",
        ticklen=ticklen,
        tickwidth=tickwidth,
        title_font_family="Arial"),
    yaxis = dict(
        dtick = 1,
        title_text = Y_title,
        title_font = {"size": title_font_size},
        tickfont = dict(family='Arial', color='black', size=tickfont_size-15),
        title_standoff = title_standoff,
        color='black',
        ticks="inside",
        ticklen=ticklen,
        tickwidth=tickwidth,
        title_font_family="Arial"))
fig.update_layout(plot_bgcolor='rgb(256,256,256)')

fig.update_xaxes(showline=True, linewidth=4, linecolor='black', mirror=True)
fig.update_yaxes(showline=True, linewidth=4, linecolor='black', mirror=True)

fig.show()

In [ ]:

X = 'WP'
unit = ''
Y = X + '-SHAP'

import plotly.express as px


title_font_size = 43
tickfont_size = 40
ticklen = 5
tickwidth = 3.5
title_standoff = 20

plot_width = 600
plot_height = 530

X_title = X + unit
Y_title = 'SHAP value'

fig = px.scatter(total, x=X, y=Y,
                 color_discrete_sequence=['636EFA', '00CC96', 'EF553B'],
                 width=plot_width, height=plot_height)

fig.update_layout(
    xaxis = dict(
        tickangle = 0,
        title_text = X_title,
        title_font = {"size": title_font_size},
        tickfont = dict(family='Arial', color='black', size=tickfont_size),
        title_standoff = title_standoff,
        color='black',
        ticks="inside",
        ticklen=ticklen,
        tickwidth=tickwidth,
        title_font_family="Arial"),
    yaxis = dict(
        dtick = 5,
        title_text = Y_title,
        title_font = {"size": title_font_size},
        tickfont = dict(family='Arial', color='black', size=tickfont_size),
        title_standoff = title_standoff,
        color='black',
        ticks="inside",
        ticklen=ticklen,
        tickwidth=tickwidth,
        title_font_family="Arial"))

fig.update_layout(plot_bgcolor='rgb(256,256,256)')

fig.update_xaxes(showline=True, linewidth=4.5, linecolor='black', mirror=True)
fig.update_yaxes(showline=True, linewidth=4.5, linecolor='black', mirror=True)

fig.update_traces(marker=dict(size=12, opacity=0.65, line=dict(width=0.7)), selector=dict(mode='markers'), showlegend=False)

fig.show()

In [ ]:
import joblib
import pickle


with open('Js_FO_ML_XGB_model.pkl', 'wb') as f:
    pickle.dump(final_model, f)

print("✅ 反向盐通量(Js)模型已成功保存为 'Js_FO_ML_XGB_model.pkl'")
print(f"模型特征数量: {final_model.n_features_in_}")